In [ ]:
from bctools.io import InstrumentResponse
from bctools.loc import LocalLocTable
from bctools.spectra.spectrum import BandFunction,Comptonized
import os
import math
import multiprocessing
import itertools
import pickle 

run_name = "run10"
irf_path = "/data/models/irf_summed_"+run_name+".h5"
dir_path = "/data/test_newrepo/"


In [ ]:
# Convolve an full instrument response with a hypothetical spectrum

# The code is inspired by the bc-tools tutorial.

import pickle

# The first time this should be False, then you can put True and speedup the loading.
load_from_file = True

if load_from_file:

    with open(dir_path+'/soft_lut_' + run_name + '.pkl', 'rb') as f:
        soft_sky_loctable = pickle.load(f)
    
    with open(dir_path+'/medium_lut_' + run_name + '.pkl', 'rb') as f:
        medium_sky_loctable = pickle.load(f)
    
    with open(dir_path+'/hard_lut_' + run_name + '.pkl', 'rb') as f:
        hard_sky_loctable = pickle.load(f)
    
else:

    with InstrumentResponse(irf_path) as irf: 
        
        # Hypothetical spectrum
        # This normalization corresponds to 1 ph/cm2/s between 50-300 keV
        soft_spectrum = BandFunction._from_megalib(['BandFunction',10,10000,-1.9,-3.7,230],"10.0")
        medium_spectrum = BandFunction._from_megalib(['BandFunction',10,10000,-1,-2.3,699.9],"10.0")
        hard_spectrum = Comptonized._from_megalib(['Comptonized',10,10000,-0.5,1500],"10.0")
        
        # In this case we integrate the rate from all energy channels. 
        # You can subdivide the data into multiple energy channel groups
        soft_local_loctable = LocalLocTable.from_irf(irf, soft_spectrum,energy_channels = 1) # [80,2000]
        medium_local_loctable = LocalLocTable.from_irf(irf, medium_spectrum,energy_channels = 1)
        hard_local_loctable = LocalLocTable.from_irf(irf, hard_spectrum,energy_channels = 1)
        
    # The local_loctable contains the expected rates in spacecraft coordinates
    # We now need to use this to estimate the total expected counts in sky coordinate for
    # the full duration of an event. 
    # In this case we simply have a 1 second event and specifying the attitude by a quaternion
    # ([0,0,0,1] corresponds to the identity rotation). You can have multiple attitude-duration
    # pairs to correctly model long duration events.
    soft_sky_loctable = soft_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    medium_sky_loctable = medium_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    hard_sky_loctable = hard_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    
    import pickle
    
    # Store the LUTS on file
    with open(dir_path+'/soft_lut_' + run_name + '.pkl', 'wb') as f:
        pickle.dump(soft_sky_loctable, f)
    with open(dir_path+'/medium_lut_' + run_name + '.pkl', 'wb') as f:
        pickle.dump(medium_sky_loctable, f)
    with open(dir_path+'/hard_lut_' + run_name + '.pkl', 'wb') as f:
        pickle.dump(hard_sky_loctable, f)

In [ ]:
hard_sky_loctable

In [ ]:

def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg

In [ ]:
print(f"Soft Look-up tables: {soft_sky_loctable.labels}")
print(f"Medium Look-up tables: {medium_sky_loctable.labels}")
print(f"Hard Look-up tables: {hard_sky_loctable.labels}")

In [ ]:
medium_sky_loctable.get_expectation_map('BGO_Z1').data

In [ ]:
if False:
    # Store the expectation map to plot it with other tools
    with open(dir_path+"/exp_medium_Y1.pkl", "wb") as f:
        pickle.dump(medium_sky_loctable.get_expectation_map('BGO_Y1').data, f)


In [ ]:
import healpy as hp
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np

def get_coord_helpix(nside, pix_id):
    
    # Parameters
    nside = 16  # Replace with your NSIDE value
    ipix = 1    # Replace with the HEALPix pixel ID (nested scheme)
    
    # Get the angular coordinates (theta, phi) of the pixel center
    theta, phi = hp.pix2ang(nside, ipix, nest=True)
    
    # Convert to equatorial coordinates (RA, Dec)
    ra = phi * 180.0 / np.pi            # phi is longitude in radians (RA)
    dec = 90.0 - theta * 180.0 / np.pi  # theta is colatitude (Dec)
    
    # Create the SkyCoord object
    coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame='icrs')

    return coord


In [ ]:
hard_sky_loctable.get_expectation_map('BGO_Z1').plot()

In [ ]:
run_name_test = "dataset"
file_name_test = "run57_mix_mega_shared"

file_path ='/data/analysis/'+run_name_test+'/'+file_name_test+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
    loaded_array_test = pickle.load(file)

In [ ]:
loaded_array_test.shape

In [ ]:
import numpy as np

filter_flux = 1
filter_spectra = 1

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] >14 and grb['flux'] <= 16:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        #if  grb['spectra'] == 'medium':
        if  '230' in grb['spectrum']:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
loaded_array_test.shape

In [ ]:
test_dataset=loaded_array_test

In [ ]:
len(test_dataset)

In [ ]:
grb_test = test_dataset[0]
[grb_test['counts'][3],grb_test['counts'][2],grb_test['counts'][5],grb_test['counts'][4],grb_test['counts'][1],grb_test['counts'][0]]

In [ ]:
grb_test['coord']

In [ ]:
grb_test

In [ ]:
process_lut_source(grb_test,None)

In [ ]:
from bctools.loc import TSMap, NormLocLike
import astropy.units as u
from astropy.coordinates import SkyCoord

b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
random_bkg = np.random.poisson(np.array([b_sim[3],b_sim[2],b_sim[5],b_sim[4],b_sim[1],b_sim[0]])*20)

def process_lut_source(grb,random_bkg):
    
    theta_real = float(grb['coord'][0])
    phi_real = float(grb['coord'][1])
    s_counts = np.array([grb['counts'][3],grb['counts'][2],grb['counts'][5],grb['counts'][4],grb['counts'][1],grb['counts'][0]])
    spectra_value = grb['spectrum']
    
    b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
    random_bkg = np.random.poisson(np.array([b_sim[3],b_sim[2],b_sim[5],b_sim[4],b_sim[1],b_sim[0]])*20)

    b_counts = np.array([b_sim[3],b_sim[2],b_sim[5],b_sim[4],b_sim[1],b_sim[0]])*20
    s_counts = s_counts+random_bkg
    
    soft_sqrt_ts, soft_ra_loc, soft_dec_loc, soft_cont_radius,soft_ts,soft_loc_tsvalue,soft_cont_area = localize_grb(soft_sky_loctable,s_counts, b_counts,theta_real,phi_real)
    medium_sqrt_ts, medium_ra_loc, medium_dec_loc, medium_cont_radius,medium_ts,medium_loc_tsvalue,medium_cont_area = localize_grb(medium_sky_loctable,s_counts, b_counts,theta_real,phi_real)
    hard_sqrt_ts, hard_ra_loc, hard_dec_loc, hard_cont_radius,hard_ts,hard_loc_tsvalue,hard_cont_area = localize_grb(hard_sky_loctable,s_counts, b_counts,theta_real,phi_real)

    max_ts = max(soft_sqrt_ts,medium_sqrt_ts,hard_sqrt_ts)
    spectra_fitted = False
    
    if max_ts == soft_sqrt_ts:
        if spectra_value == "soft":
            spectra_fitted = True
        original_ts_value = soft_loc_tsvalue
        best_ts = soft_ts
        ra_loc=soft_ra_loc
        dec_loc=soft_dec_loc
        cont_radius = soft_cont_radius
        cont_area = soft_cont_area
    
    elif max_ts == medium_sqrt_ts:
        if spectra_value == "medium":
            spectra_fitted = True
        original_ts_value = medium_loc_tsvalue
        best_ts = medium_ts
        ra_loc=medium_ra_loc
        dec_loc=medium_dec_loc
        cont_radius = medium_cont_radius
        cont_area = medium_cont_area
    
    elif max_ts == hard_sqrt_ts:
        if spectra_value == "hard":
            spectra_fitted = True
        original_ts_value = hard_loc_tsvalue
        best_ts = hard_ts
        ra_loc=hard_ra_loc
        dec_loc=hard_dec_loc
        cont_radius = hard_cont_radius
        cont_area = hard_cont_area
    else:
        print("ts not found")
        print(grb)
    
    theta_loc, phi_loc = ra_dec_to_theta_phi(ra_loc, dec_loc)

    dist = angular_distance(theta_loc, phi_loc, theta_real, phi_real)
    theta_dist = np.abs(theta_loc - theta_real)
    phi_dist = diff_phi(phi_loc, phi_real)

    result = [theta_real, phi_real, theta_loc, phi_loc, dist, theta_dist, phi_dist, max_ts, cont_radius,best_ts,original_ts_value,cont_area,spectra_fitted]
    
    return result
    


def localize_grb(sky_loctable,s,b,theta_real,phi_real):

    
    sky_loctable.set_background(b)
    sky_loctable.set_data(s)

    # Define a map of nside = 32. Note that this is a finer resolution
    #that the underlying look-up table, which will be interpolated
    ts = TSMap(nside = 64, coordsys = 'icrs')

    # NormLocLike is a subclass of LocLike and computes
    # a Poisson likelihood for counting instruments. The
    # overall normalization is the only free parameter
    norm_likelihood = NormLocLike(sky_loctable)
   
    # Compute the TS map from one or more LocLikelihood
    ts.compute(norm_likelihood)
    #print("#"+str(theta_real)+"#")
    
    # Correggi theta_real prima di convertirlo
    if theta_real < 0:
        print(f"Invalid theta_real: {theta_real}")
        theta_real = 0
    elif theta_real > 180:
        print(f"Invalid theta_real: {theta_real}")
        theta_real = 180
    
    theta_rad = np.deg2rad(theta_real)
    phi_rad = np.deg2rad(phi_real)
    
    if theta_rad < 0 or theta_rad > np.pi:
        print(f"theta_rad fuori range: {theta_rad}")


    #print(dir(ts))
    ipix = ts.ang2pix(theta_rad, phi_rad)

    # Get the value at the given coordinates
    #ipix = ts.ang2pix(ts.nside, theta_real * u.deg.to(u.rad), phi_real * u.deg.to(u.rad))
    original_ts_value = ts._data[ipix]

    confidence_levels = np.arange(0, 1.01, 0.01)
    
    containment_radius = np.array([
        np.sqrt(ts.error_area(cont=cont) / np.pi).to(u.deg).value
        for cont in confidence_levels
    ])
  
    #original_ts_value = -1
    cont_area = ts.error_area(cont = .9).to(u.deg**2).value

    

    return np.max(ts),ts.best_loc().ra.deg,ts.best_loc().dec.deg,containment_radius,ts,original_ts_value,cont_area

def ra_dec_to_theta_phi(ra, dec):

    theta = 90-dec
    
    phi = ra
    
    return theta, phi

def diff_phi(a1, a2):
    # Calcola la differenza diretta
    diff = abs(a1 - a2)
    
    # Trova il percorso più breve tenendo conto del ciclo degli angoli
    if diff > 180:
        diff = 360 - diff
    
    return diff

In [ ]:
results_bkg = []
spectra_fitted = 0


def init_worker():
    seed = int.from_bytes(os.urandom(4), "little")
    np.random.seed(seed)


def process_in_parallel(test_dataset,random_bkg):
    with multiprocessing.Pool(processes=200, initializer=init_worker) as pool:
        results = pool.starmap(process_lut_source, zip(test_dataset, itertools.repeat(random_bkg)))
    return results

import math ,time
print(time.time())
results_bkg = process_in_parallel(test_dataset,random_bkg)
print(time.time())

In [ ]:
distances = []
theta_distances = []
phi_distances = []
cont_radius_list = []
cont_area_list = []
for res in results_bkg:
    distances.append(res[4])
    theta_distances.append(res[5])
    phi_distances.append(res[6])
    cont_radius_list.append(res[8])
    cont_area_list.append(res[11])

In [ ]:
print(np.mean(distances))
print(np.mean(cont_area_list))

In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

nside = 32
npix = hp.nside2npix(nside)


m = np.array(cont_area_list)

hp.projview(
    m,
    coord=["G"],
    graticule=True,
    graticule_labels=True,
    unit="deg2",
    xlabel="longitude",
    ylabel="latitude",
    cb_orientation="vertical",
    latitude_grid_spacing=30,
    projection_type="aitoff",
    title="Aitoff projection",
    cmap="turbo",
    nest=True
)

plt.show()


In [ ]:
import pickle
if False:

    # Salva l'array in un file usando pickle
    with open(dir_path+"/bc_"+file_name_test+"_distall.pkl", "wb") as f:
        pickle.dump(distances, f)
        
    # Salva l'array in un file usando pickle
    with open(dir_path+"/bc_"+file_name_test+"_cont_area.pkl", "wb") as f:
        pickle.dump(cont_area_list, f)

In [ ]:
test_labels = []
for grb in test_dataset:
    theta_real = float(grb['coord'][0])
    phi_real = float(grb['coord'][1])
    test_labels.append([theta_real,phi_real])

test_labels = np.array(test_labels)

In [ ]:
print(np.mean(distances))
print(np.mean(theta_distances))
print(np.mean(phi_distances))
print(np.mean(cont_area_list))

In [ ]:
# Define the 5-degree intervals
intervals = np.arange(0, 185, 5)

# Group counts based on 5-degree intervals
grouped_counts = np.zeros((len(intervals)))
counts = np.zeros((len(intervals)))

tot_count = 0
for theta, phi, count in zip(test_labels[:,0], test_labels[:,1], cont_area_list):
    
    if(True): #(phi>125 and phi<145) or 
        tot_count += 1
        interval_index = int(theta // 5)
        grouped_counts[interval_index] = grouped_counts[interval_index] + count
        counts[interval_index] += 1

# Compute the average counts in each interval
grouped_counts = grouped_counts[:tot_count]
counts = counts[:tot_count]

# Display the histogram
plt.figure(figsize=(10, 6))
plt.bar(intervals[:-1], grouped_counts[:-1]/counts[:-1], width=5, align='edge', alpha=1, label="loc. error")
plt.xlabel('Theta (°)')
plt.ylabel('Loc. error')
plt.title('Loc. error as a function of theta')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
step = 10

# Define the 5-degree intervals
intervals = np.arange(0, 365, step)

# Group counts based on 5-degree intervals
grouped_counts_1 = np.zeros((len(intervals)))
counts_1 = np.zeros((len(intervals)))

grouped_counts_2 = np.zeros((len(intervals)))
counts_2 = np.zeros((len(intervals)))

grouped_counts_3 = np.zeros((len(intervals)))
counts_3 = np.zeros((len(intervals)))

grouped_counts_4 = np.zeros((len(intervals)))
counts_4 = np.zeros((len(intervals)))

total_count_1 = 0
total_count_2 = 0
total_count_3 = 0
total_count_4 = 0

for theta, phi, count in zip(test_labels[:,0], test_labels[:,1], distances):
    
    if(theta > 20 and theta < 55):
        total_count_1 += 1
        interval_index = int(phi // step)
        grouped_counts_1[interval_index] = grouped_counts_1[interval_index] + count
        counts_1[interval_index] += 1

    if(theta > 55 and theta < 150):
        total_count_2 += 1
        interval_index = int(phi // step)
        grouped_counts_2[interval_index] = grouped_counts_2[interval_index] + count
        counts_2[interval_index] += 1

    if(theta > 150 and theta < 180):
        total_count_3 += 1
        interval_index = int(phi // step)
        grouped_counts_3[interval_index] = grouped_counts_3[interval_index] + count
        counts_3[interval_index] += 1

# Compute the average counts in each interval
grouped_counts_1 = grouped_counts_1[:total_count_1]
counts_1 = counts_1[:total_count_1]

grouped_counts_2 = grouped_counts_2[:total_count_2]
counts_2 = counts_2[:total_count_2]

grouped_counts_3 = grouped_counts_3[:total_count_3]
counts_3 = counts_3[:total_count_3]

# Display the histogram
plt.figure(figsize=(10, 6))
plt.bar(intervals[:-1], grouped_counts_1[:-1]/counts_1[:-1], width=step, align='edge', alpha=0.5, label="loc. error theta=[20°,55°]")

plt.bar(intervals[:-1], grouped_counts_2[:-1]/counts_2[:-1], width=step, align='edge', alpha=0.5, label="loc. error theta=[55°,150°]")

plt.bar(intervals[:-1], grouped_counts_3[:-1]/counts_3[:-1], width=step, align='edge', alpha=0.5, label="loc. error theta=[150°,180°]")

plt.xlabel('Phi (°)')
plt.ylabel('Loc. error')
plt.title('Loc. error as a function of phi')
plt.grid(True)
plt.legend()
plt.show()
